# MSRV3 — Itens de pedido de compra — Análise Exploratória

**Tabela:** `dev_procurement.corp_curated.tbl_ds_pro_msrv3`
**Transação SAP:** MSRV3 · **Objeto:** **Tabela física**
**Colunas:** 23 · **Clustering declarado:** `cod_organizacao_compras`, `cod_centro`

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente. Não há widget, view temporária nem ordem obrigatória entre elas.

## Alertas de partida _(extraídos dos comentários do `DESCRIBE`)_

| Severidade | Alerta |
|---|---|
| 🔴 ALTA | **Divergência de nomenclatura:** no SAP a MSRV3 é *folha de registro de serviços*, mas todas as colunas desta tabela são de **pedido de compra** (EBELN, EBELP, EKORG, EKGRP). Confirmar com o time se o de/para transação↔tabela está correto. |
| 🟡 MEDIA | `ind_eliminado` e `st_processamento` são **derivados** — validar a regra (seção 18.1). |
| ⚪ BAIXA | Única das 5 tabelas que é **tabela física** (as outras 4 são views). |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria |
| 3 | Distribuição por centro — escolha do cenário |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação com o SAP** |
| 12 | Datas e timestamps |
| 13 | Códigos e zeros à esquerda |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| **18** | **Análises específicas da transação** |
| 19 | Amostra ampliada |
| 20 | Resumo |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_pro_msrv3;

In [0]:
-- Formato, tamanho e particoes
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_pro_msrv3;

In [0]:
-- Ultimas gravacoes
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_msrv3 LIMIT 20;

## 2. Volumetria

In [0]:
-- 2. VOLUMETRIA
SELECT COUNT(*)                                        AS linhas,
       COUNT(DISTINCT CONCAT_WS('|', `num_pedido_compra`, `num_item_pedido_compra`, `dateingest`)) AS chaves_completas,
       COUNT(DISTINCT `cod_centro`)                 AS valores_de_cod_centro,
       COUNT(DISTINCT `num_pedido_compra`)                 AS num_pedido_compra_distintos
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3;

## 3. Distribuição por centro

Base para escolher o cenário de teste: recorte com volume viável (10 mil a 300 mil linhas)
que contenha os casos-limite identificados nas demais seções.

In [0]:
-- 3. DISTRIBUICAO POR cod_centro
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)') AS `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 50;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional multiplicando linhas.

In [0]:
-- GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3),
g AS (
  SELECT 'num_pedido_compra' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3)
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3)
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra + dateingest' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra`, `dateingest` FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3)
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `cod_organizacao_compras + cod_centro`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `cod_organizacao_compras`, `cod_centro`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
GROUP BY `cod_organizacao_compras`, `cod_centro`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [0]:
-- O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH dup AS (
  SELECT `cod_organizacao_compras`, `cod_centro` FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 GROUP BY `cod_organizacao_compras`, `cod_centro` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 c JOIN dup ON c.`cod_organizacao_compras` <=> dup.`cod_organizacao_compras` AND c.`cod_centro` <=> dup.`cod_centro`
),
agg AS (
  SELECT `cod_organizacao_compras`, `cod_centro`,
         COUNT(DISTINCT `num_pedido_compra`) AS `num_pedido_compra`,
         COUNT(DISTINCT `num_item_pedido_compra`) AS `num_item_pedido_compra`,
         COUNT(DISTINCT `cod_grupo_compradores`) AS `cod_grupo_compradores`,
         COUNT(DISTINCT `cod_fornecedor`) AS `cod_fornecedor`,
         COUNT(DISTINCT `dt_pedido_compra`) AS `dt_pedido_compra`,
         COUNT(DISTINCT `cod_moeda_documento`) AS `cod_moeda_documento`,
         COUNT(DISTINCT `cod_eliminacao_documento`) AS `cod_eliminacao_documento`,
         COUNT(DISTINCT `st_completude_compra`) AS `st_completude_compra`,
         COUNT(DISTINCT `cod_usuario_criacao`) AS `cod_usuario_criacao`,
         COUNT(DISTINCT `dt_criacao`) AS `dt_criacao`,
         COUNT(DISTINCT `desc_item_pedido`) AS `desc_item_pedido`,
         COUNT(DISTINCT `qt_pedido`) AS `qt_pedido`,
         COUNT(DISTINCT `sg_unidade_pedido`) AS `sg_unidade_pedido`,
         COUNT(DISTINCT `cod_grupo_mercadoria`) AS `cod_grupo_mercadoria`,
         COUNT(DISTINCT `vl_bruto`) AS `vl_bruto`,
         COUNT(DISTINCT `vl_liquido`) AS `vl_liquido`,
         COUNT(DISTINCT `ind_eliminado`) AS `ind_eliminado`,
         COUNT(DISTINCT `st_processamento`) AS `st_processamento`,
         COUNT(DISTINCT `dateingest`) AS `dateingest`,
         COUNT(DISTINCT `yearingest`) AS `yearingest`,
         COUNT(DISTINCT `monthingest`) AS `monthingest`
  FROM d GROUP BY `cod_organizacao_compras`, `cod_centro`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(21,
    'num_pedido_compra', MAX(`num_pedido_compra`),
    'num_item_pedido_compra', MAX(`num_item_pedido_compra`),
    'cod_grupo_compradores', MAX(`cod_grupo_compradores`),
    'cod_fornecedor', MAX(`cod_fornecedor`),
    'dt_pedido_compra', MAX(`dt_pedido_compra`),
    'cod_moeda_documento', MAX(`cod_moeda_documento`),
    'cod_eliminacao_documento', MAX(`cod_eliminacao_documento`),
    'st_completude_compra', MAX(`st_completude_compra`),
    'cod_usuario_criacao', MAX(`cod_usuario_criacao`),
    'dt_criacao', MAX(`dt_criacao`),
    'desc_item_pedido', MAX(`desc_item_pedido`),
    'qt_pedido', MAX(`qt_pedido`),
    'sg_unidade_pedido', MAX(`sg_unidade_pedido`),
    'cod_grupo_mercadoria', MAX(`cod_grupo_mercadoria`),
    'vl_bruto', MAX(`vl_bruto`),
    'vl_liquido', MAX(`vl_liquido`),
    'ind_eliminado', MAX(`ind_eliminado`),
    'st_processamento', MAX(`st_processamento`),
    'dateingest', MAX(`dateingest`),
    'yearingest', MAX(`yearingest`),
    'monthingest', MAX(`monthingest`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada.

Em validação anterior, esta análise revelou 7 colunas 100% nulas no Datalake —
uma delas preenchida em 97,9% dos registros do SAP. Este erro **não aparece** em amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

In [0]:
-- PREENCHIMENTO DE TODAS AS COLUNAS
-- Para boolean, a coluna 'zeros_ou_false' conta os valores FALSE.
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3),
perf AS (
  SELECT stack(23,
    'num_pedido_compra', 'string', COUNT_IF(`num_pedido_compra` IS NULL), COUNT_IF(`num_pedido_compra` IS NOT NULL AND lower(trim(`num_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'num_item_pedido_compra', 'string', COUNT_IF(`num_item_pedido_compra` IS NULL), COUNT_IF(`num_item_pedido_compra` IS NOT NULL AND lower(trim(`num_item_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'cod_organizacao_compras', 'string', COUNT_IF(`cod_organizacao_compras` IS NULL), COUNT_IF(`cod_organizacao_compras` IS NOT NULL AND lower(trim(`cod_organizacao_compras`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_organizacao_compras`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_compradores', 'string', COUNT_IF(`cod_grupo_compradores` IS NULL), COUNT_IF(`cod_grupo_compradores` IS NOT NULL AND lower(trim(`cod_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_compradores`) RLIKE '^0+([.,]0+)?$'),
    'cod_fornecedor', 'string', COUNT_IF(`cod_fornecedor` IS NULL), COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'dt_pedido_compra', 'date', COUNT_IF(`dt_pedido_compra` IS NULL), 0L, 0L,
    'cod_moeda_documento', 'string', COUNT_IF(`cod_moeda_documento` IS NULL), COUNT_IF(`cod_moeda_documento` IS NOT NULL AND lower(trim(`cod_moeda_documento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda_documento`) RLIKE '^0+([.,]0+)?$'),
    'cod_eliminacao_documento', 'string', COUNT_IF(`cod_eliminacao_documento` IS NULL), COUNT_IF(`cod_eliminacao_documento` IS NOT NULL AND lower(trim(`cod_eliminacao_documento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_eliminacao_documento`) RLIKE '^0+([.,]0+)?$'),
    'st_completude_compra', 'string', COUNT_IF(`st_completude_compra` IS NULL), COUNT_IF(`st_completude_compra` IS NOT NULL AND lower(trim(`st_completude_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_completude_compra`) RLIKE '^0+([.,]0+)?$'),
    'cod_usuario_criacao', 'string', COUNT_IF(`cod_usuario_criacao` IS NULL), COUNT_IF(`cod_usuario_criacao` IS NOT NULL AND lower(trim(`cod_usuario_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_usuario_criacao`) RLIKE '^0+([.,]0+)?$'),
    'dt_criacao', 'date', COUNT_IF(`dt_criacao` IS NULL), 0L, 0L,
    'desc_item_pedido', 'string', COUNT_IF(`desc_item_pedido` IS NULL), COUNT_IF(`desc_item_pedido` IS NOT NULL AND lower(trim(`desc_item_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_item_pedido`) RLIKE '^0+([.,]0+)?$'),
    'qt_pedido', 'decimal(13,3)', COUNT_IF(`qt_pedido` IS NULL), 0L, COUNT_IF(`qt_pedido` = 0),
    'sg_unidade_pedido', 'string', COUNT_IF(`sg_unidade_pedido` IS NULL), COUNT_IF(`sg_unidade_pedido` IS NOT NULL AND lower(trim(`sg_unidade_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_pedido`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_mercadoria', 'string', COUNT_IF(`cod_grupo_mercadoria` IS NULL), COUNT_IF(`cod_grupo_mercadoria` IS NOT NULL AND lower(trim(`cod_grupo_mercadoria`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_mercadoria`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'vl_bruto', 'double', COUNT_IF(`vl_bruto` IS NULL), 0L, COUNT_IF(`vl_bruto` = 0),
    'vl_liquido', 'double', COUNT_IF(`vl_liquido` IS NULL), 0L, COUNT_IF(`vl_liquido` = 0),
    'ind_eliminado', 'string', COUNT_IF(`ind_eliminado` IS NULL), COUNT_IF(`ind_eliminado` IS NOT NULL AND lower(trim(`ind_eliminado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_eliminado`) RLIKE '^0+([.,]0+)?$'),
    'st_processamento', 'string', COUNT_IF(`st_processamento` IS NULL), COUNT_IF(`st_processamento` IS NOT NULL AND lower(trim(`st_processamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_processamento`) RLIKE '^0+([.,]0+)?$'),
    'dateingest', 'date', COUNT_IF(`dateingest` IS NULL), 0L, 0L,
    'yearingest', 'string', COUNT_IF(`yearingest` IS NULL), COUNT_IF(`yearingest` IS NOT NULL AND lower(trim(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`yearingest`) RLIKE '^0+([.,]0+)?$'),
    'monthingest', 'string', COUNT_IF(`monthingest` IS NULL), COUNT_IF(`monthingest` IS NOT NULL AND lower(trim(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`monthingest`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros_ou_false)
  FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros_ou_false,
       t.total - p.nulos - p.vazios - p.zeros_ou_false                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros_ou_false) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                               THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros_ou_false <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros_ou_false) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                                      AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Coluna constante é candidata a valor default de carga.

In [0]:
-- CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3),
card AS (
  SELECT stack(23,
    'num_pedido_compra', 'string', approx_count_distinct(`num_pedido_compra`),
    'num_item_pedido_compra', 'string', approx_count_distinct(`num_item_pedido_compra`),
    'cod_organizacao_compras', 'string', approx_count_distinct(`cod_organizacao_compras`),
    'cod_grupo_compradores', 'string', approx_count_distinct(`cod_grupo_compradores`),
    'cod_fornecedor', 'string', approx_count_distinct(`cod_fornecedor`),
    'dt_pedido_compra', 'date', approx_count_distinct(`dt_pedido_compra`),
    'cod_moeda_documento', 'string', approx_count_distinct(`cod_moeda_documento`),
    'cod_eliminacao_documento', 'string', approx_count_distinct(`cod_eliminacao_documento`),
    'st_completude_compra', 'string', approx_count_distinct(`st_completude_compra`),
    'cod_usuario_criacao', 'string', approx_count_distinct(`cod_usuario_criacao`),
    'dt_criacao', 'date', approx_count_distinct(`dt_criacao`),
    'desc_item_pedido', 'string', approx_count_distinct(`desc_item_pedido`),
    'qt_pedido', 'decimal(13,3)', approx_count_distinct(`qt_pedido`),
    'sg_unidade_pedido', 'string', approx_count_distinct(`sg_unidade_pedido`),
    'cod_grupo_mercadoria', 'string', approx_count_distinct(`cod_grupo_mercadoria`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'vl_bruto', 'double', approx_count_distinct(`vl_bruto`),
    'vl_liquido', 'double', approx_count_distinct(`vl_liquido`),
    'ind_eliminado', 'string', approx_count_distinct(`ind_eliminado`),
    'st_processamento', 'string', approx_count_distinct(`st_processamento`),
    'dateingest', 'date', approx_count_distinct(`dateingest`),
    'yearingest', 'string', approx_count_distinct(`yearingest`),
    'monthingest', 'string', approx_count_distinct(`monthingest`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma.

In [0]:
-- DOMINIO DAS CATEGORICAS
(SELECT 'cod_organizacao_compras' AS coluna, CAST(`cod_organizacao_compras` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 GROUP BY `cod_organizacao_compras` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_compradores' AS coluna, CAST(`cod_grupo_compradores` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 GROUP BY `cod_grupo_compradores` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda_documento' AS coluna, CAST(`cod_moeda_documento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 GROUP BY `cod_moeda_documento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_eliminacao_documento' AS coluna, CAST(`cod_eliminacao_documento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 GROUP BY `cod_eliminacao_documento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_completude_compra' AS coluna, CAST(`st_completude_compra` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 GROUP BY `st_completude_compra` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_eliminado' AS coluna, CAST(`ind_eliminado` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 GROUP BY `ind_eliminado` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_processamento' AS coluna, CAST(`st_processamento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 GROUP BY `st_processamento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_unidade_pedido' AS coluna, CAST(`sg_unidade_pedido` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 GROUP BY `sg_unidade_pedido` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_mercadoria' AS coluna, CAST(`cod_grupo_mercadoria` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 GROUP BY `cod_grupo_mercadoria` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'yearingest' AS coluna, CAST(`yearingest` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 GROUP BY `yearingest` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 10. Perfil dos campos numéricos

A tabela tem **2 colunas `double`** entre as numéricas analisadas — exigem tolerância de 0,005 na comparação com o SAP.

In [0]:
-- PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(3,
    'qt_pedido', 'decimal(13,3)', COUNT(`qt_pedido`), CAST(MIN(`qt_pedido`) AS DOUBLE), CAST(MAX(`qt_pedido`) AS DOUBLE), CAST(AVG(`qt_pedido`) AS DOUBLE), CAST(percentile_approx(`qt_pedido`, 0.5) AS DOUBLE), COUNT_IF(`qt_pedido` < 0), COUNT_IF(`qt_pedido` = 0),
    'vl_bruto', 'double', COUNT(`vl_bruto`), CAST(MIN(`vl_bruto`) AS DOUBLE), CAST(MAX(`vl_bruto`) AS DOUBLE), CAST(AVG(`vl_bruto`) AS DOUBLE), CAST(percentile_approx(`vl_bruto`, 0.5) AS DOUBLE), COUNT_IF(`vl_bruto` < 0), COUNT_IF(`vl_bruto` = 0),
    'vl_liquido', 'double', COUNT(`vl_liquido`), CAST(MIN(`vl_liquido`) AS DOUBLE), CAST(MAX(`vl_liquido`) AS DOUBLE), CAST(AVG(`vl_liquido`) AS DOUBLE), CAST(percentile_approx(`vl_liquido`, 0.5) AS DOUBLE), COUNT_IF(`vl_liquido` < 0), COUNT_IF(`vl_liquido` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel e compare. Divergência de total detecta registro
faltando ou duplicado — cobre o ponto cego da contagem de linhas.

In [0]:
-- TOTAIS PARA CONCILIACAO COM O SAP
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(3,
    'qt_pedido', CAST(SUM(`qt_pedido`) AS DOUBLE), CAST(ROUND(SUM(`qt_pedido`), 2) AS STRING), COUNT(`qt_pedido`),
    'vl_bruto', CAST(SUM(`vl_bruto`) AS DOUBLE), CAST(ROUND(SUM(`vl_bruto`), 2) AS STRING), COUNT(`vl_bruto`),
    'vl_liquido', CAST(SUM(`vl_liquido`) AS DOUBLE), CAST(ROUND(SUM(`vl_liquido`), 2) AS STRING), COUNT(`vl_liquido`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
)
ORDER BY coluna;

## 12. Datas em tipo nativo

In [0]:
-- DATAS EM TIPO NATIVO
SELECT * FROM (
  SELECT stack(3,
    'dt_pedido_compra', COUNT_IF(`dt_pedido_compra` IS NULL), CAST(MIN(`dt_pedido_compra`) AS STRING), CAST(MAX(`dt_pedido_compra`) AS STRING), COUNT(DISTINCT `dt_pedido_compra`), COUNT_IF(`dt_pedido_compra` > current_date()),
    'dt_criacao', COUNT_IF(`dt_criacao` IS NULL), CAST(MIN(`dt_criacao`) AS STRING), CAST(MAX(`dt_criacao`) AS STRING), COUNT(DISTINCT `dt_criacao`), COUNT_IF(`dt_criacao` > current_date()),
    'dateingest', COUNT_IF(`dateingest` IS NULL), CAST(MIN(`dateingest`) AS STRING), CAST(MAX(`dateingest`) AS STRING), COUNT(DISTINCT `dateingest`), COUNT_IF(`dateingest` > current_date())
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras)
  FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
)
ORDER BY coluna;

## 13. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- CODIGOS: ZEROS A ESQUERDA E FORMATO
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(6,
    'num_pedido_compra', 'string', COUNT_IF(CAST(`num_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_pedido_compra` AS STRING)), '^0+', '')),
    'num_item_pedido_compra', 'string', COUNT_IF(CAST(`num_item_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_item_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_item_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_pedido_compra` AS STRING)), '^0+', '')),
    'cod_fornecedor', 'string', COUNT_IF(CAST(`cod_fornecedor` AS STRING) IS NULL OR trim(CAST(`cod_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cod_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cod_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_fornecedor` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_organizacao_compras', 'string', COUNT_IF(CAST(`cod_organizacao_compras` AS STRING) IS NULL OR trim(CAST(`cod_organizacao_compras` AS STRING)) = ''), MIN(length(trim(CAST(`cod_organizacao_compras` AS STRING)))), MAX(length(trim(CAST(`cod_organizacao_compras` AS STRING)))), COUNT_IF(trim(CAST(`cod_organizacao_compras` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_organizacao_compras` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_organizacao_compras` AS STRING)), '^0+', '')),
    'cod_grupo_compradores', 'string', COUNT_IF(CAST(`cod_grupo_compradores` AS STRING) IS NULL OR trim(CAST(`cod_grupo_compradores` AS STRING)) = ''), MIN(length(trim(CAST(`cod_grupo_compradores` AS STRING)))), MAX(length(trim(CAST(`cod_grupo_compradores` AS STRING)))), COUNT_IF(trim(CAST(`cod_grupo_compradores` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_grupo_compradores` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_grupo_compradores` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
)
ORDER BY coluna;

## 14. Checksum de linha

Se `linhas` > `linhas_unicas`, existem registros 100% idênticos — duplicata real,
não granularidade adicional.

In [0]:
-- CHECKSUM DE LINHA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`cod_organizacao_compras` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`dt_pedido_compra` AS STRING), ''), COALESCE(CAST(`cod_moeda_documento` AS STRING), ''), COALESCE(CAST(`cod_eliminacao_documento` AS STRING), ''), COALESCE(CAST(`st_completude_compra` AS STRING), ''), COALESCE(CAST(`cod_usuario_criacao` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`desc_item_pedido` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`sg_unidade_pedido` AS STRING), ''), COALESCE(CAST(`cod_grupo_mercadoria` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`vl_bruto` AS STRING), ''), COALESCE(CAST(`vl_liquido` AS STRING), ''), COALESCE(CAST(`ind_eliminado` AS STRING), ''), COALESCE(CAST(`st_processamento` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))            AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`cod_organizacao_compras` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`dt_pedido_compra` AS STRING), ''), COALESCE(CAST(`cod_moeda_documento` AS STRING), ''), COALESCE(CAST(`cod_eliminacao_documento` AS STRING), ''), COALESCE(CAST(`st_completude_compra` AS STRING), ''), COALESCE(CAST(`cod_usuario_criacao` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`desc_item_pedido` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`sg_unidade_pedido` AS STRING), ''), COALESCE(CAST(`cod_grupo_mercadoria` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`vl_bruto` AS STRING), ''), COALESCE(CAST(`vl_liquido` AS STRING), ''), COALESCE(CAST(`ind_eliminado` AS STRING), ''), COALESCE(CAST(`st_processamento` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), '')))) AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`cod_organizacao_compras` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`dt_pedido_compra` AS STRING), ''), COALESCE(CAST(`cod_moeda_documento` AS STRING), ''), COALESCE(CAST(`cod_eliminacao_documento` AS STRING), ''), COALESCE(CAST(`st_completude_compra` AS STRING), ''), COALESCE(CAST(`cod_usuario_criacao` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`desc_item_pedido` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`sg_unidade_pedido` AS STRING), ''), COALESCE(CAST(`cod_grupo_mercadoria` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`vl_bruto` AS STRING), ''), COALESCE(CAST(`vl_liquido` AS STRING), ''), COALESCE(CAST(`ind_eliminado` AS STRING), ''), COALESCE(CAST(`st_processamento` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''), COALESCE(CAST(`yearingest` AS STRING), ''), COALESCE(CAST(`monthingest` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3;

## 15. Amostra de linhas completas

In [0]:
-- 15. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
ORDER BY `num_pedido_compra`, `num_item_pedido_compra`
LIMIT 20;

## 16. Distribuição interna

Como o volume se reparte nas principais dimensões.

In [0]:
-- 16. DISTRIBUICAO POR cod_centro
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)') AS `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_organizacao_compras
SELECT COALESCE(NULLIF(trim(CAST(`cod_organizacao_compras` AS STRING)), ''), '(vazio)') AS `cod_organizacao_compras`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_organizacao_compras` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_grupo_compradores
SELECT COALESCE(NULLIF(trim(CAST(`cod_grupo_compradores` AS STRING)), ''), '(vazio)') AS `cod_grupo_compradores`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_grupo_compradores` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR st_processamento
SELECT COALESCE(NULLIF(trim(CAST(`st_processamento` AS STRING)), ''), '(vazio)') AS `st_processamento`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
GROUP BY COALESCE(NULLIF(trim(CAST(`st_processamento` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

In [0]:
-- 17. FRESHNESS
SELECT `dateingest` AS data_carga, COUNT(*) AS linhas
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
GROUP BY `dateingest`
ORDER BY data_carga DESC
LIMIT 30;

In [0]:
-- 17.1 SNAPSHOT OU HISTORICO?
SELECT MIN(`dateingest`)            AS primeira_carga,
       MAX(`dateingest`)            AS ultima_carga,
       COUNT(DISTINCT `dateingest`) AS cargas_distintas,
       CASE WHEN COUNT(DISTINCT `dateingest`) = 1
            THEN 'SNAPSHOT - substitui a cada carga; chave NAO precisa da data'
            ELSE 'HISTORICO - acumula; a chave DEVE incluir a data de carga' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3;

## 18. Análises específicas — MSRV3

### 18.1 Validação das colunas derivadas

O schema declara duas derivações:

- `ind_eliminado` = `X` quando `cod_eliminacao_documento` = `L`
- `st_processamento` = `COMPLETED` quando `st_completude_compra` = `C`, senão `OPEN`

Esta célula confirma se a regra foi aplicada corretamente em todas as linhas.

In [0]:
-- 18.1 VALIDACAO DE ind_eliminado
SELECT cod_eliminacao_documento,
       ind_eliminado,
       COUNT(*) AS linhas,
       CASE WHEN cod_eliminacao_documento = 'L' AND ind_eliminado = 'X' THEN 'ok'
            WHEN (cod_eliminacao_documento IS NULL OR cod_eliminacao_documento <> 'L')
             AND (ind_eliminado IS NULL OR trim(ind_eliminado) = '')      THEN 'ok'
            ELSE 'DIVERGENTE DA REGRA' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
GROUP BY cod_eliminacao_documento, ind_eliminado
ORDER BY linhas DESC;

In [0]:
-- 18.1b VALIDACAO DE st_processamento
SELECT st_completude_compra,
       st_processamento,
       COUNT(*) AS linhas,
       CASE WHEN st_completude_compra = 'C' AND st_processamento = 'COMPLETED' THEN 'ok'
            WHEN (st_completude_compra IS NULL OR st_completude_compra <> 'C')
             AND st_processamento = 'OPEN'                                      THEN 'ok'
            ELSE 'DIVERGENTE DA REGRA' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
GROUP BY st_completude_compra, st_processamento
ORDER BY linhas DESC;

### 18.2 Coerência entre valor bruto e líquido

O valor bruto deveria ser maior ou igual ao líquido.

In [0]:
-- 18.2 VALOR BRUTO x LIQUIDO
SELECT COUNT(*)                                            AS linhas,
       COUNT_IF(vl_bruto >= vl_liquido)                    AS bruto_maior_ou_igual,
       COUNT_IF(vl_bruto < vl_liquido)                     AS bruto_MENOR_que_liquido,
       COUNT_IF(vl_bruto = vl_liquido)                     AS iguais,
       COUNT_IF(vl_bruto = 0 AND vl_liquido = 0)           AS ambos_zerados,
       COUNT_IF(qt_pedido = 0)                             AS quantidade_zerada,
       COUNT_IF(qt_pedido < 0)                             AS quantidade_negativa,
       ROUND(SUM(vl_bruto), 2)                             AS soma_bruto,
       ROUND(SUM(vl_liquido), 2)                           AS soma_liquido
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3;

### 18.3 Itens por pedido e coerência temporal

In [0]:
-- 18.3 ITENS POR PEDIDO
SELECT itens_no_pedido, COUNT(*) AS pedidos
FROM (SELECT num_pedido_compra, COUNT(*) AS itens_no_pedido
        FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3 GROUP BY num_pedido_compra)
GROUP BY itens_no_pedido
ORDER BY itens_no_pedido
LIMIT 40;

In [0]:
-- 18.3b COERENCIA TEMPORAL (datas nativas)
SELECT COUNT(*)                                           AS linhas,
       COUNT_IF(dt_criacao > dt_pedido_compra)            AS criacao_apos_pedido,
       COUNT_IF(dt_criacao = dt_pedido_compra)            AS mesma_data,
       COUNT_IF(dt_pedido_compra IS NULL)                 AS pedido_sem_data,
       MIN(dt_pedido_compra)                              AS pedido_mais_antigo,
       MAX(dt_pedido_compra)                              AS pedido_mais_recente,
       MIN(dt_criacao)                                    AS criacao_mais_antiga,
       MAX(dt_criacao)                                    AS criacao_mais_recente
FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3;

## 19. Amostra ampliada

O notebook exploratório não extrai a base completa. Para isso, use um notebook de cenário
com o recorte escolhido na seção 3.

In [0]:
-- 19. AMOSTRA AMPLIADA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3
ORDER BY `num_pedido_compra`, `num_item_pedido_compra`
LIMIT 500;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3),
g AS (
  SELECT 'num_pedido_compra' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3)
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3)
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra + dateingest' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra`, `dateingest` FROM dev_procurement.corp_curated.tbl_ds_pro_msrv3)
)
SELECT 'IDENTIFICACAO' AS bloco, 'transacao' AS item, 'MSRV3' AS valor
UNION ALL SELECT 'IDENTIFICACAO', 'tabela', 'tbl_ds_pro_msrv3'
UNION ALL SELECT 'IDENTIFICACAO', 'tipo de objeto', 'TABELA'
UNION ALL SELECT 'IDENTIFICACAO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'IDENTIFICACAO', 'colunas', '23'
UNION ALL SELECT 'IDENTIFICACAO', 'coluna de recorte', 'cod_centro'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 3**.
2. Gerar o notebook de cenário com esse recorte.
3. Extrair a MSRV3 no SAP com o mesmo recorte e na **mesma data** do snapshot.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Checklist antes de comparar

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 13)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 10)
- [ ] Totais numéricos conferidos (seção 11)
- [ ] Alertas de partida verificados (seção 18)
